# V031: protected local adapter before the unchanged Laplace body
Only span3 temporal context changes. Exact frozen V024 adapter math; unchanged V029 mean/scale heads, likelihood, curve penalty, preprocessing and native-MAE selection. Definitions are inert.

In [ ]:
"""Only a protected zero-initialized span3 adapter is added to frozen V029."""
import copy,hashlib,json
from pathlib import Path
import numpy as np
import torch
from torch import nn

V031_ARCHITECTURE=dict(version='v031-protected-span3-before-v029-v1',span=3,offsets=[-1,0,1],
    activation='SiLU',weight_shape='[3,n_features,n_features]',initialization='all zero, inherited V029 body constructed first, no extra RNG',
    protected_numeric_outputs=['R','C','u_out','time_step','step'],protected_internal_joint_code=True,
    padding='zero within each independent80-step breath; no wrap or cross-breath context',
    body='unchanged V029PressureLaplaceRNN mean and positive-scale heads',
    source_math='frozen V024TemporalResidualRNN shifted/context_values/adapt_inputs methods',
    objective='unchanged V029_OBJECTIVE: inspiratory Laplace NLL + .1 adjacent mean-curve loss')

class V031PressureLaplaceConvRNN(V029PressureLaplaceRNN):
    def __init__(self,spec,n_features,feature_names):
        super().__init__(spec,n_features)
        self.span=int(V031_ARCHITECTURE['span']);self.feature_names=list(feature_names)
        if self.span!=3 or len(self.feature_names)!=self.n_features or len(set(self.feature_names))!=self.n_features:
            raise ValueError('V031 permits only span3 and a unique complete feature schema')
        if not {'R','C','u_out','time_step','step'}.issubset(self.feature_names):raise ValueError('Known lung/phase/time channels must be named')
        self.context_weight=nn.Parameter(torch.zeros(self.span,self.n_features,self.n_features))
        mask=torch.tensor([0. if n in {'R','C','u_out','time_step','step'} else 1. for n in self.feature_names])
        self.register_buffer('context_output_mask',mask.reshape(1,1,-1))
        self.offsets=tuple(range(-self.span//2+1,self.span//2+1))
        if tuple(self.offsets)!=(-1,0,1):raise RuntimeError('V031 temporal offsets changed')
    shifted=staticmethod(V024TemporalResidualRNN.shifted)
    context_values=V024TemporalResidualRNN.context_values
    adapt_inputs=V024TemporalResidualRNN.adapt_inputs
    def forward(self,x):return super().forward(self.adapt_inputs(x))

def v031_conv3_laplace_factory(spec,n_features,preprocess):
    return V031PressureLaplaceConvRNN(spec,n_features,preprocess['feature_names'])

def v031_frozen_conv3_laplace_factory(spec,n_features,preprocess):
    model=v031_conv3_laplace_factory(spec,n_features,preprocess);model.context_weight.requires_grad_(False);return model

def _v031_metadata(arm,loss_kind,source_fingerprints):
    if loss_kind!='laplace_curve':raise ValueError('V031 retains actual loss_kind laplace_curve')
    sources=dict(source_fingerprints or {});actual={}
    for name in ['convolution_components_v024.ipynb','laplace_components_v029.ipynb','laplace_convolution_components_v031.ipynb']:
        digest=hashlib.sha256(_v029_code(_v8_notebook_path(name)).encode()).hexdigest()
        if sources.get(name)!=digest:raise ValueError('Actual V031 inherited/model source pin required: '+name)
        actual[name]=digest
    return dict(adapter_version='v031-isolated-convolution-laplace-engine-v1',arm=arm,
        architecture=copy.deepcopy(V031_ARCHITECTURE),objective=copy.deepcopy(V029_OBJECTIVE),component_code_sha256=actual)

def _v031_engine(arm,loss_kind,model_factory,source_fingerprints,device):
    architecture=_v031_metadata(arm,loss_kind,source_fingerprints)
    namespace,compat,objective=_v029_engine(arm,loss_kind,model_factory,source_fingerprints,device)
    old_fingerprint=namespace['_fingerprint'];old_payload=namespace['_checkpoint_payload']
    def fingerprint(*args,**kwargs):
        result=old_fingerprint(*args,**kwargs);result['v031']=copy.deepcopy(architecture);return result
    def payload(*args,**kwargs):
        result=old_payload(*args,**kwargs);result['v031']=copy.deepcopy(architecture)
        result['actual_model_architecture']='protected_span3_before_v029';return result
    namespace.update(_fingerprint=fingerprint,_checkpoint_payload=payload)
    return namespace,compat,objective,architecture

def _v031_report(report,compat,objective,architecture):
    result=_v029_report(report,compat,objective)
    result.update(v031=copy.deepcopy(architecture),actual_model_architecture='protected_span3_before_v029')
    return result

def fit_validate_v031(x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,
        epoch_log_path=None,feature_names=None,train_diagnostic_breaths=2048,device='cuda',resume=False,on_epoch=None,*,
        model_factory=v031_conv3_laplace_factory,loss_kind='laplace_curve',arm='v031_conv3_laplace',source_fingerprints=None):
    namespace,compat,objective,architecture=_v031_engine(arm,loss_kind,model_factory,source_fingerprints,device)
    report=namespace['fit_validate'](x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,
        epoch_log_path=epoch_log_path,feature_names=feature_names,train_diagnostic_breaths=train_diagnostic_breaths,device=device,resume=resume,on_epoch=on_epoch)
    return _v031_report(report,compat,objective,architecture)

def fit_fixed_v031(x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,
        epoch_log_path=None,feature_names=None,device='cuda',resume=False,on_epoch=None,*,model_factory=v031_conv3_laplace_factory,
        loss_kind='laplace_curve',arm='v031_conv3_laplace',source_fingerprints=None):
    namespace,compat,objective,architecture=_v031_engine(arm,loss_kind,model_factory,source_fingerprints,device)
    report=namespace['fit_fixed'](x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,
        epoch_log_path=epoch_log_path,feature_names=feature_names,device=device,resume=resume,on_epoch=on_epoch)
    return _v031_report(report,compat,objective,architecture)

def load_checkpoint_v031(checkpoint_path,device='cpu',*,model_factory=v031_conv3_laplace_factory,expected_arm=None,
        expected_loss_kind=None,source_fingerprints=None):
    if expected_loss_kind not in (None,'laplace_curve'):raise ValueError('V031 actual loss remains laplace_curve')
    saved=torch.load(Path(checkpoint_path),map_location='cpu',weights_only=False)
    if 'v031' not in saved or saved.get('actual_model_architecture')!='protected_span3_before_v029':raise ValueError('Expected truthful V031 architecture checkpoint')
    arm=saved['v031']['arm']
    if expected_arm is not None and arm!=expected_arm:raise ValueError('V031 arm changed')
    architecture=_v031_metadata(arm,'laplace_curve',source_fingerprints)
    if saved['v031']!=architecture or saved['fingerprint'].get('v031')!=architecture:raise ValueError('V031 architecture/source changed')
    return load_checkpoint_v029(checkpoint_path,device=device,model_factory=model_factory,expected_arm=arm,
        expected_loss_kind='laplace_curve',source_fingerprints=source_fingerprints)

@torch.no_grad()
def predict_checkpoint_laplace_v031(checkpoint_path,x,device='cpu',batch_size=512,*,return_scale=False,
        model_factory=v031_conv3_laplace_factory,expected_arm=None,expected_loss_kind=None,source_fingerprints=None):
    if int(batch_size)<1:raise ValueError('Positive batch size required')
    model,prep,saved=load_checkpoint_v031(checkpoint_path,device=device,model_factory=model_factory,
        expected_arm=expected_arm,expected_loss_kind=expected_loss_kind,source_fingerprints=source_fingerprints)
    transformed=transform_inputs(x,prep);output=np.empty(transformed.shape[:2],dtype=np.float32);scales=np.empty_like(output)
    for start in range(0,len(transformed),int(batch_size)):
        batch=torch.from_numpy(transformed[start:start+batch_size]).to(_sequence_device(device))
        output[start:start+batch_size]=model(batch).cpu().numpy();scales[start:start+batch_size]=model.v029_scale.cpu().numpy();model.clear_scale_v029()
    codes=transformed[:,0,-1].astype(np.int64)
    output=output*prep['target_scale']+prep['target_center']+prep['baseline_profile'][codes];scales=scales*prep['target_scale']
    if not np.isfinite(output).all() or not np.isfinite(scales).all() or not (scales>0).all():raise RuntimeError('Invalid native mean/scale')
    output=output.astype(np.float32,copy=False);scales=scales.astype(np.float32,copy=False)
    return (output,scales) if return_scale else output

def predict_checkpoint_v031(checkpoint_path,x,device='cpu',batch_size=512,**kwargs):
    return predict_checkpoint_laplace_v031(checkpoint_path,x,device=device,batch_size=batch_size,return_scale=False,**kwargs)

def v031_workflow_namespace(project_root):
    """Versioned private V025 queue; NLL objective unchanged, architecture explicit."""
    namespace=v029_workflow_namespace(project_root);original_cohort=namespace['pruning_cohort_v025']
    def cohort(*args,**kwargs):
        result=original_cohort(*args,**kwargs);result['architecture']=copy.deepcopy(V031_ARCHITECTURE);return result
    def completed_prediction(outcome,x,*,model_factory=None,device='cpu',batch_size=512):
        if outcome['status']!='completed':raise ValueError('Pruned/interrupted model cannot supply a completed candidate')
        meta=outcome['report']['v031']
        return predict_checkpoint_v031(outcome['checkpoint_file'],x,device=device,batch_size=batch_size,
            model_factory=v031_conv3_laplace_factory if model_factory is None else model_factory,expected_arm=meta['arm'],
            expected_loss_kind='laplace_curve',source_fingerprints=outcome['source_fingerprints'])
    namespace.update(fit_validate_v008=fit_validate_v031,pruning_cohort_v025=cohort,predict_completed_v025=completed_prediction)
    return namespace
